# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. All explorations reference entities (record sets, fields, columns) by their Croissant `@id` fields as per best practice.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`, referencing dataset structure by their `@id`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display metadata overview
meta = dataset.metadata
print(f"Dataset name: {getattr(meta, 'name', None)}\nDescription: {getattr(meta, 'description', None)}\nVersion: {getattr(meta, 'version', None)}\nIdentifier: {getattr(meta, 'identifier', None)}\nLicense: {getattr(meta, 'license', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entity references use their `@id`.

The `mlcroissant` library provides entities such as RecordSets, with hierarchical Field definitions. Let's enumerate record sets and their fields programmatically.

In [ ]:
# List available RecordSets by `@id` and their fields
recordsets = dataset.record_sets
pp = pprint.PrettyPrinter(indent=2)
print(f"Number of record sets: {len(recordsets)}\n")
record_set_ids = []
for rs in recordsets:
    print(f"RecordSet name: {rs.name}\n  @id: {rs.id}\n  description: {rs.description}")
    record_set_ids.append(rs.id)
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print("")

if not record_set_ids:
    print("No record sets found in the schema. Please check the schema or documentation.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the above overview to ensure all operations are by `@id`.

Below, we extract all record sets found above. If there is only one record set, only that will be used.

In [ ]:
# Prepare a list of record_set IDs to extract
if record_set_ids:
    selected_record_sets = record_set_ids
else:
    selected_record_sets = []

dataframes = {}
for rs_id in selected_record_sets:
    print(f"Loading records for RecordSet @id={rs_id} ...")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for {rs_id} with shape {df.shape}")
        print(f"Fields (columns) in this set: {df.columns.tolist()}")
    else:
        print(f"No records loaded for {rs_id}")
if dataframes:
    # Preview first loaded DataFrame
    first_rs_id = next(iter(dataframes))
    display(dataframes[first_rs_id].head())
else:
    print("No dataframes loaded. Check schema/data access.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records, normalizing numeric fields, and grouping/categorizing data. All processing steps should reference fields by their `@id`.

Let's select a likely numeric field for EDA. If no numeric fields detected, instructions are provided for user input.

In [ ]:
import numpy as np

# Attempt to find a numeric field automatically from the first recordset
numeric_field = None
group_field = None
first_rs_id = None
if dataframes:
    first_rs_id = next(iter(dataframes))
    df = dataframes[first_rs_id]
    # Try to auto-identify numeric columns
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    # Try a likely group field - fallback to the second column
    if len(df.columns) > 1:
        group_field = df.columns[1]
    else:
        group_field = None

if numeric_field is not None:
    print(f"Using numeric field '@id': {numeric_field}")
    threshold = df[numeric_field].quantile(0.75)  # use a percentile value
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records where {numeric_field} > {threshold:.2f} (top quartile)")
    print(filtered_df.head())

    # Normalize the selected numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nFirst 5 values for normalized '{numeric_field}':")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    if group_field is not None:
        print(f"\nGrouping by field '@id': {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(by=numeric_field, ascending=False)
        print(grouped_df.head())
else:
    print("No numeric fields detected automatically. Please explore the columns and specify a numeric field for analysis by its `@id`.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Grouped bar plots, histograms, or scatter plots are common EDA tools.

Below, we plot a histogram of the selected numeric field and (if a group field exists) a bar plot of means by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Histogram of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

    if group_field is not None:
        # Show a boxplot by group if group_field is likely categorical
        plt.figure(figsize=(10, 4))
        if df[group_field].nunique() < 20:
            sns.boxplot(x=group_field, y=numeric_field, data=df)
            plt.title(f"{numeric_field} by {group_field}")
            plt.tight_layout()
            plt.show()
else:
    print("No visualization performed because no numeric field was detected.")

## 6. Conclusion
This notebook demonstrated end-to-end dataset exploration using Croissant `@id` referencing for the FAIR² dataset. Key findings include:
- Croissant schemas enable programmatic, standards-based data exploration.
- All steps are reproducible and referenced via globally unique entity IDs.
- Example EDA included filtering, normalization, grouping, and visualizations by field `@id`s.

For further domain-specific analysis, consult the [dataset homepage](https://sen.science/doi/10.71728/senscience.vq4a-28xa) or extend this notebook with custom feature engineering.
